# Pipeline 7: Incident Risk Prediction

**Organization:** River of Life / Lighthouse Sanctuary (INTEX)  
**Methodology:** CRISP-DM–aligned (see `pipeline_guide.md` in this folder)

---

## Executive Summary

We estimate the **probability of at least one safety or wellbeing incident** in the **next 60 days** per resident-month snapshot, using **only** prior case history and recent education/health signals. Outputs are **internal decision support** for supervision and safety huddles—not a substitute for mandatory reporting or professional judgment.

**What this notebook delivers**
- **Rigorous preprocessing:** imputation and scaling inside **`sklearn.Pipeline`**, fit on training folds only (no global leakage).
- **Grouped validation** by `resident_id` (panel rows are correlated).
- **Predictive model:** Random Forest (and dummy baseline) for ranking and recall-focused operations.
- **Explanatory model:** **Balanced logistic regression** for signed associations and staff-facing narratives.
- **Rich EDA**, **multi-threshold evaluation**, **feature-importance synthesis**, and an **expanded causal / limitations** discussion aligned with INTEX expectations.

*Non-technical readers:* skim the Executive Summary, **Business Interpretation**, **Key Findings**, and **Recommended Actions**, then use Sections 3–7 for discussion with data and program staff.

---



## 1. Problem Framing

### Business problem
Incidents are often managed **reactively**. Case managers, house parents, and safety leads need a **data-grounded watchlist** of residents whose **recent record pattern** resembles months that later saw incidents—so teams can **prioritize check-ins, staffing, and prevention** without waiting for the next event.

### Stakeholders
| Stakeholder | Interest |
|-------------|----------|
| **House parents / social workers** | Daily supervision and documentation load |
| **Safety officer** | Briefings, huddles, escalation |
| **Leadership** | Aggregate risk governance and staffing |

### Why this matters
Earlier, **proportionate** attention—guided by record patterns—may **reduce harm** when combined with training and policy. The model does **not** replace human judgment or legal reporting duties.

### Predictive goal (60-day horizon)
**Binary target:** whether the resident experiences **≥1 incident** in **`(as_of_date, as_of_date + 60 days]`** (see Data Validity).  
**Deliverable:** a **risk score** suitable for **ranking** and **threshold-based** flagging given finite supervision time.

### Explanatory goal
Understand **which observable factors co-occur** with elevated 60-day incident risk (e.g., prior incident load, health/education signals, site, case phase)—for **training**, **quality improvement**, and **ethical review** of the scorecard.

### Predictive vs explanatory (explicit)
| Role | Model | Why |
|------|-------|-----|
| **Predictive (performance)** | **Random Forest** | Captures **nonlinearities** and **interactions** (e.g., high prior incidents × low health). |
| **Explanatory (interpretability)** | **Balanced logistic regression** | **Coefficients** on a comparable feature space after preprocessing. |
| **Baseline** | **Stratified dummy** | Sanity check—scores must **beat chance** aligned to prevalence. |

### Success metrics
- **ROC-AUC:** ranking quality for watchlists.
- **Recall:** catching future incidents (safety-oriented); trade off against **precision** / staff load.
- **Comparison to dummy** on grouped CV every time the pipeline is refreshed.

### Decision supported
**Weekly case conference:** review **top-risk** resident-months (internal list), aligned to **operational threshold** (Section 5), not a single fixed 0.5 cutoff.

### Limitations (preview)
Rare positive class, **observational** data, and **documentation bias**—expanded in Section 7.

---



## Data Validity & Leakage Check

### How the target is defined
**`y = 1`** if any `incident_reports` row for the resident has **`incident_date` in (`as_of_date`, `as_of_date + 60` days]**.

### What information is allowed at prediction time
- **Incidents** in the **prior 90 days** ending at `as_of_date` (count, severity, unresolved count)—**not** incidents after `as_of_date`.
- **Education** and **health** rows with `record_date` in the same **90-day** window.
- **Resident static / slow-moving fields** at `as_of_date`: risk level, case status, safehouse (encoded as categorical).

### Why future information does not leak into features
The label uses only **`incident_date > as_of_date`**. Feature engineering uses **`<= as_of_date`** for all event tables.

### Preprocessing and leakage
**Median imputation** and **scaling** run **inside** the sklearn **`Pipeline`**, so each **GroupKFold** training split learns statistics **only from that fold’s training rows**. We **do not** impute missing education/health means using the **full panel** before splitting.

### Why grouped train/test is valid
The same `resident_id` appears on **many monthly snapshots**. **`GroupShuffleSplit`** (holdout) and **`GroupKFold`** (CV) **keep all rows of a resident** in one split, avoiding optimistic metrics.

### Automated checks
The notebook prints panel size, **positive rate**, and compares models **on the training resident set** for CV, then reports **held-out residents** for calibration-style checks.

---



## 2. Data Acquisition & Preparation

### Tables
| Table | Use |
|-------|-----|
| `residents` | `safehouse_id`, `current_risk_level`, `case_status`, `date_of_admission` → **tenure** |
| `incident_reports` | **Target** (future window) and **prior 90d** incident load, max severity, unresolved count |
| `education_records` | Rolling **mean / std / count / trend** of `progress_percent` (90d) |
| `health_wellbeing_records` | Rolling **mean / std / count** of `general_health_score` (90d) |

### Feature definitions (why each matters for incident risk)
| Feature | Meaning | Rationale |
|---------|---------|-----------|
| `incidents_90d` | Count of prior-window incidents | **Persistence:** past instability predicts future events. |
| `max_sev_90d` | Worst severity (ordinal 1–3) in window | Captures **acute** episodes, not just frequency. |
| `unresolved_incidents_90d` | Count still open at snapshot | Ongoing safety issues may elevate near-term risk. |
| `health_mean_90d` / `health_std_90d` | Level and **volatility** of wellbeing scores | **Deterioration or instability** may co-occur with crises. |
| `health_count_90d` | # health touchpoints | Engagement vs sparse documentation. |
| `edu_prog_mean_90d` / `edu_prog_std_90d` | Level and **volatility** of academic progress | Stress or disengagement may surface in school records. |
| `edu_count_90d` | # education records | Data availability / engagement proxy. |
| `edu_prog_trend_90d` | Average monthly change in progress in window | **Trajectory** beyond the mean. |
| `tenure_days` | Days since admission to `as_of_date` | Early placement vs longer-stay dynamics. |
| `safehouse_id` (categorical) | Site | **Context:** staffing, environment, cohort mix (**not** blame). |
| `risk_lvl`, `status` | Clinical risk tier and case phase | Stratification; interpret with **confounding** caution (Section 7). |

### Categorical encoding
**`safehouse_id`** is treated as **categorical** (string labels + **one-hot**), not as a numeric magnitude.

---



In [ ]:
import json
import warnings
from datetime import timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=UserWarning)
RANDOM_STATE = 42
HORIZON_DAYS = 60
PRIOR_WINDOW_DAYS = 90
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("talk", font_scale=0.85)


def resolve_data_and_artifacts() -> tuple[Path, Path]:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        ml = p / "machine_learning" / "lighthouse_csv_v7"
        if ml.is_dir():
            return ml, p / "machine_learning" / "ml_pipelines" / "artifacts"
        root = p / "lighthouse_csv_v7"
        if root.is_dir():
            return root, p / "ml_pipelines" / "artifacts"
    raise FileNotFoundError("lighthouse_csv_v7 not found (try machine_learning/lighthouse_csv_v7).")


DATA_DIR, OUTPUT_DIR = resolve_data_and_artifacts()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
_base = OUTPUT_DIR.parent.parent
print("DATA_DIR:", DATA_DIR.resolve().relative_to(_base.resolve()))
print("OUTPUT_DIR:", OUTPUT_DIR.resolve().relative_to(_base.resolve()))
print("HORIZON_DAYS:", HORIZON_DAYS)



In [ ]:
SEVERITY_MAP = {"Low": 1, "Medium": 2, "High": 3}


def build_incident_panel(
    residents: pd.DataFrame,
    inc: pd.DataFrame,
    edu: pd.DataFrame,
    hl: pd.DataFrame,
    anchor_start: str = "2023-06-01",
    anchor_end: str = "2025-10-01",
) -> pd.DataFrame:
    rows: list[dict] = []
    anchors = pd.date_range(anchor_start, anchor_end, freq="MS")
    mx = inc["incident_date"].max()
    admit = residents.set_index("resident_id")["date_of_admission"]

    for as_of in anchors:
        if as_of + timedelta(days=HORIZON_DAYS) > mx:
            continue
        d0 = as_of - timedelta(days=PRIOR_WINDOW_DAYS)
        for rid in residents["resident_id"].unique():
            fut = inc[
                (inc["resident_id"] == rid)
                & (inc["incident_date"] > as_of)
                & (inc["incident_date"] <= as_of + timedelta(days=HORIZON_DAYS))
            ]
            y = int(len(fut) > 0)

            past = inc[
                (inc["resident_id"] == rid)
                & (inc["incident_date"] >= d0)
                & (inc["incident_date"] <= as_of)
            ]
            n_past = len(past)
            if n_past:
                sev = past["severity"].map(SEVERITY_MAP)
                mx_sev = float(sev.max()) if sev.notna().any() else 0.0
                unres = int((~past["resolved"].fillna(True)).sum())
            else:
                mx_sev = 0.0
                unres = 0

            hp = hl[
                (hl["resident_id"] == rid)
                & (hl["record_date"] >= d0)
                & (hl["record_date"] <= as_of)
            ]
            h_count = int(len(hp))
            h_mean = float(hp["general_health_score"].mean()) if h_count else np.nan
            h_std = float(hp["general_health_score"].std()) if h_count >= 2 else np.nan

            ep = edu[
                (edu["resident_id"] == rid)
                & (edu["record_date"] >= d0)
                & (edu["record_date"] <= as_of)
            ].sort_values("record_date")
            e_count = int(len(ep))
            pr_mean = float(ep["progress_percent"].mean()) if e_count else np.nan
            pr_std = float(ep["progress_percent"].std()) if e_count >= 2 else np.nan
            if e_count >= 2:
                pr_trend = float(
                    (ep.iloc[-1]["progress_percent"] - ep.iloc[0]["progress_percent"])
                    / max(e_count - 1, 1)
                )
            else:
                pr_trend = 0.0

            r = residents[residents["resident_id"] == rid].iloc[0]
            adm = admit.get(rid)
            tenure_days = int((as_of - adm).days) if pd.notna(adm) else np.nan

            rows.append(
                {
                    "resident_id": rid,
                    "as_of_date": as_of,
                    "y": y,
                    "incidents_90d": n_past,
                    "max_sev_90d": mx_sev,
                    "unresolved_incidents_90d": unres,
                    "health_mean_90d": h_mean,
                    "health_std_90d": h_std,
                    "health_count_90d": h_count,
                    "edu_prog_mean_90d": pr_mean,
                    "edu_prog_std_90d": pr_std,
                    "edu_count_90d": e_count,
                    "edu_prog_trend_90d": pr_trend,
                    "tenure_days": tenure_days,
                    "safehouse_id": str(int(r["safehouse_id"])),
                    "risk_lvl": r["current_risk_level"],
                    "status": r["case_status"],
                }
            )
    return pd.DataFrame(rows)


res = pd.read_csv(DATA_DIR / "residents.csv", parse_dates=["date_of_admission"])
inc = pd.read_csv(DATA_DIR / "incident_reports.csv", parse_dates=["incident_date"])
edu = pd.read_csv(DATA_DIR / "education_records.csv", parse_dates=["record_date"])
hl = pd.read_csv(DATA_DIR / "health_wellbeing_records.csv", parse_dates=["record_date"])

p = build_incident_panel(res, inc, edu, hl)
print("Panel rows:", len(p), "  Positive rate (y=1):", round(p["y"].mean(), 4))
p.head()



## 3. Exploration (EDA)

Each chart below is followed by a short interpretation (**what it suggests** and **how it informed modeling**). EDA uses the **full panel** for **marginal distributions** and **rates by segment**—final model fitting uses **pipeline preprocessing trained only on training data** (Section 4).

---



In [ ]:
eda = p.copy()

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
vc = eda["y"].value_counts().reindex([0, 1], fill_value=0)
ax[0].bar(["No incident\n(next 60d)", "≥1 incident\n(next 60d)"], vc.values, color=["#66bd63", "#d73027"])
ax[0].set_ylabel("Resident-month rows")
ax[0].set_title("Target distribution")
eda["y"].value_counts(normalize=True).plot(kind="bar", ax=ax[1], color=["#66bd63", "#d73027"], rot=0)
ax[1].set_title("Class share")
ax[1].set_ylabel("Proportion")
plt.tight_layout()
plt.show()

eda["prior_bin"] = pd.cut(
    eda["incidents_90d"],
    bins=[-0.01, 0, 1, 2, 4, 500],
    labels=["0", "1", "2", "3–4", "5+"],
)
rate_by_prior = eda.groupby("prior_bin", observed=True)["y"].mean()
cnt_by_prior = eda.groupby("prior_bin", observed=True).size()
plt.figure(figsize=(8, 4))
plt.bar(range(len(rate_by_prior)), rate_by_prior.values, color="#8073ac")
plt.xticks(range(len(rate_by_prior)), list(rate_by_prior.index.astype(str)))
plt.ylabel("P(incident in next 60d)")
plt.xlabel("Prior incidents in last 90d (binned)")
plt.title("Future incident rate vs prior incident load")
plt.tight_layout()
plt.show()

rl = eda.groupby("risk_lvl")["y"].agg(["mean", "count"]).sort_values("mean", ascending=False)
plt.figure(figsize=(8, 4))
plt.bar(rl.index.astype(str), rl["mean"], color="#d95f02")
plt.ylabel("Incident rate (next 60d)")
plt.xlabel("current_risk_level")
plt.title("Observed incident rate by intake/clinical risk label")
plt.tight_layout()
plt.show()

st = eda.groupby("status")["y"].agg(["mean", "count"])
plt.figure(figsize=(8, 4))
plt.bar(st.index.astype(str), st["mean"], color="#1b7837")
plt.ylabel("Incident rate")
plt.xticks(rotation=25, ha="right")
plt.title("Incident rate by case_status")
plt.tight_layout()
plt.show()

sh = eda.groupby("safehouse_id")["y"].agg(["mean", "count"]).sort_values("mean", ascending=False)
plt.figure(figsize=(9, 4))
plt.bar(sh.index.astype(str), sh["mean"], color="#2166ac")
plt.ylabel("Incident rate")
plt.xlabel("safehouse_id")
plt.title("Incident rate by safehouse (categorical context)")
plt.tight_layout()
plt.show()

_ten = eda["tenure_days"].fillna(eda["tenure_days"].median())
eda["tenure_bin"] = pd.qcut(_ten.clip(lower=0), q=4, duplicates="drop")
tt = eda.groupby("tenure_bin", observed=True)["y"].mean()
plt.figure(figsize=(9, 4))
tt.plot(kind="bar", color="#762a83", rot=30)
plt.ylabel("Incident rate")
plt.title("Incident rate by tenure quartile (proxy for time in program)")
plt.tight_layout()
plt.show()

miss = pd.Series(
    {
        "edu_prog_mean_90d": eda["edu_prog_mean_90d"].isna().mean(),
        "health_mean_90d": eda["health_mean_90d"].isna().mean(),
        "edu_prog_std_90d": eda["edu_prog_std_90d"].isna().mean(),
        "health_std_90d": eda["health_std_90d"].isna().mean(),
        "tenure_days": eda["tenure_days"].isna().mean(),
    }
)
plt.figure(figsize=(8, 4))
miss.sort_values().plot(kind="barh", color="#a65628")
plt.xlabel("Share of rows missing")
plt.title("Missingness (no records in 90d window → NaN before pipeline imputation)")
plt.tight_layout()
plt.show()

num_cols = [
    "incidents_90d",
    "max_sev_90d",
    "unresolved_incidents_90d",
    "health_mean_90d",
    "edu_prog_mean_90d",
    "tenure_days",
    "edu_prog_trend_90d",
]
cm = eda[num_cols + ["y"]].corr(numeric_only=True, min_periods=15)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Correlation heatmap (numeric features + target)")
plt.tight_layout()
plt.show()



### EDA interpretations (per chart)

**Target distribution**  
The positive class is typically a **minority**. That implies **class-weighted** models, **ROC-AUC / recall** over naive accuracy, and **threshold tuning**—a default 0.5 cutoff is usually misaligned with safety and staffing tradeoffs.

**Prior incidents vs future incidents**  
Rates rise sharply with **more prior-window incidents**, supporting **persistence of risk** and justifying **`incidents_90d`**, **`max_sev_90d`**, and **`unresolved_incidents_90d`** as core features (still **not** causal—see Section 7).

**Risk level**  
If **clinical risk labels** align with higher observed incident rates, the model can **respect** existing triage; if misaligned, the model may **add signal** from behavior in records—either way, **`risk_lvl`** is kept as **categorical context** with careful interpretation.

**Case status**  
**Active vs reintegrated/closed** phases differ in exposure and supervision intensity; **`status`** belongs in the model as a **phase** control, with awareness of **selection** (who remains Active).

**Safehouse variation**  
Differences across sites often reflect **cohort mix, staffing, or documentation**, not “bad” locations. Encoding **`safehouse_id`** as **categorical** absorbs site baselines without treating ID as a **numeric dose**.

**Tenure**  
Quartile trends hint whether **early stabilization** vs **long-stay fatigue** patterns exist; **`tenure_days`** captures a simple **time-in-program** proxy for case conferences.

**Missingness**  
Missing education/health aggregates mean **no rows in the 90d window** (or undefined std with one row). **Median imputation inside the pipeline** avoids using **test-set** information; high missingness in a segment may **bias** apparent relationships—note in deployment monitoring.

**Heatmap**  
**Prior incident count** and **severity** usually dominate linear associations with **`y`**; other correlations may be **modest**. The forest model still **combines** weaker signals.

---



## 4. Modeling & Feature Selection

### Model roles (why each one)
| Model | Purpose |
|-------|---------|
| **Dummy (stratified)** | **Baseline:** predicts at random respecting class frequency. Any serious model must beat this on **grouped AUC**. |
| **Logistic regression (balanced)** | **Explanatory:** **signed coefficients** after scaling—for ethics reviews, training, and “what tends to co-occur with risk.” |
| **Random Forest** | **Predictive:** **nonlinear** boundaries and **interactions**; typically stronger **ranking** for watchlists. |

### Preprocessing (leakage-safe)
All numeric features pass **`SimpleImputer(strategy='median')` → `StandardScaler`**. Categoricals use **`OneHotEncoder(handle_unknown='ignore')`**. The **`ColumnTransformer`** lives in a **`Pipeline`** before the classifier so **CV folds** only **fit** imputation on **training** split rows.

### Grouped cross-validation
**`GroupKFold(5)`** on **`resident_id`** using **training panel rows only** (after **`GroupShuffleSplit`** holdout). This estimates how well models **generalize to new residents**.

---



In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    classification_report,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.base import clone

FEATURE_NUM = [
    "incidents_90d",
    "max_sev_90d",
    "unresolved_incidents_90d",
    "health_mean_90d",
    "health_std_90d",
    "health_count_90d",
    "edu_prog_mean_90d",
    "edu_prog_std_90d",
    "edu_count_90d",
    "edu_prog_trend_90d",
    "tenure_days",
]
FEATURE_CAT = ["risk_lvl", "status", "safehouse_id"]

X = p[FEATURE_NUM + FEATURE_CAT]
y = p["y"]
g = p["resident_id"]

_num = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]
)
preprocess = ColumnTransformer(
    [
        ("num", _num, FEATURE_NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), FEATURE_CAT),
    ]
)


def make_pipe(est):
    return Pipeline([("prep", preprocess), ("clf", est)])


gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
tr_idx, te_idx = next(gss.split(X, y, g))
X_train, X_test = X.iloc[tr_idx], X.iloc[te_idx]
y_train, y_test = y.iloc[tr_idx], y.iloc[te_idx]
g_train = g.iloc[tr_idx]

print("Train rows", len(X_train), "residents", g_train.nunique())
print("Test rows", len(X_test), "residents", g.iloc[te_idx].nunique())



In [ ]:
gkf = GroupKFold(n_splits=5)
scoring = ["roc_auc", "recall", "precision", "f1"]
models = {
    "dummy_stratified": DummyClassifier(strategy="stratified", random_state=RANDOM_STATE),
    "logistic_balanced": LogisticRegression(
        max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE, C=1.0
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=350,
        max_depth=6,
        min_samples_leaf=6,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

cv_rows = []
for name, est in models.items():
    pipe = make_pipe(est)
    sc = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=gkf,
        groups=g_train,
        scoring=scoring,
        n_jobs=-1,
    )
    row = {"model": name}
    for m in scoring:
        row[f"{m}_mean"] = sc[f"test_{m}"].mean()
        row[f"{m}_std"] = sc[f"test_{m}"].std()
    cv_rows.append(row)

cv_table = pd.DataFrame(cv_rows).set_index("model")
cv_table.round(4)



In [ ]:
print(cv_table.round(4).to_string())

log_pipe = make_pipe(models["logistic_balanced"])
log_pipe.fit(X_train, y_train)
rf_pipe = make_pipe(models["random_forest"])
rf_pipe.fit(X_train, y_train)

proba_log = log_pipe.predict_proba(X_test)[:, 1]
proba_rf = rf_pipe.predict_proba(X_test)[:, 1]

print("\nHeld-out residents — ROC-AUC")
print("  Logistic:", round(roc_auc_score(y_test, proba_log), 4))
print("  Random Forest:", round(roc_auc_score(y_test, proba_rf), 4))

thresholds = [0.3, 0.5, 0.7]
rows_th = []
for t in thresholds:
    pred_l = (proba_log >= t).astype(int)
    pred_r = (proba_rf >= t).astype(int)
    for name, pred, pr in [
        ("logistic", pred_l, proba_log),
        ("random_forest", pred_r, proba_rf),
    ]:
        prec, rec, f1, _ = precision_recall_fscore_support(
            y_test, pred, average="binary", zero_division=0
        )
        n_flag = int(pred.sum())
        rows_th.append(
            {
                "model": name,
                "threshold": t,
                "flagged": n_flag,
                "flagged_pct": n_flag / len(y_test),
                "precision": prec,
                "recall": rec,
                "f1": f1,
            }
        )

th_df = pd.DataFrame(rows_th)
print("\nThreshold tradeoffs (test set):")
print(th_df.round(4).to_string(index=False))



In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))

t_plot = 0.5
ConfusionMatrixDisplay.from_predictions(y_test, (proba_rf >= t_plot).astype(int), ax=axes[0, 0])
axes[0, 0].set_title(f"RF confusion @ threshold {t_plot}")

RocCurveDisplay.from_predictions(y_test, proba_rf, ax=axes[0, 1])
axes[0, 1].set_title("ROC — Random Forest (held-out residents)")

ConfusionMatrixDisplay.from_predictions(y_test, (proba_log >= t_plot).astype(int), ax=axes[1, 0])
axes[1, 0].set_title(f"Logistic confusion @ threshold {t_plot}")

RocCurveDisplay.from_predictions(y_test, proba_log, ax=axes[1, 1])
axes[1, 1].set_title("ROC — Logistic (held-out residents)")

plt.tight_layout()
plt.show()



## 5. Evaluation & Interpretation

### How to read the CV table
- **Mean ± std** across **5 grouped folds** on the **training resident-months** only.
- **Lift over `dummy_stratified`:** if **AUC** is near **0.5** and **recall** mirrors prevalence, the usable signal is **weak**—treat scores as **experimental**.

### Held-out test (new residents)
**ROC-AUC** on **held-out `resident_id`s** approximates **deployment** to residents not seen in training rows. **Logistic** and **RF** are both reported: **RF** for **operations**, **logistic** for **narrative**.

### Threshold analysis (operations)
The table at **0.3 / 0.5 / 0.7** shows **how many rows would be flagged** and the **precision–recall** tradeoff:
- **Lower threshold** → **more flags**, higher **recall** (fewer missed incidents), more **false positives** (staff time, resident sensitivity).
- **Higher threshold** → fewer flags, higher **precision** among flagged, **more false negatives** (missed risk).

Tune the cutoff to **weekly huddle capacity** and policy (e.g. prefer **extra** safe false positives vs **missing** a rare event).

### Error costs (plain language)
| Error | Consequence | Mitigation |
|-------|-------------|------------|
| **False positive** | Resident discussed though no incident would occur → **staff time**, possible **stigma** if handled carelessly | Gratitude-first framing; **never** show raw scores to youth |
| **False negative** | Incident occurs without flag → **safety gap** | Lower threshold when staffing allows; **never** disable **mandatory** reporting |

---



## Business Interpretation

### What this means in plain English
Scores surface residents whose **documentation pattern** resembles months that later had incidents. That is **pattern recognition**, not a verdict on a child.

### How reliable for real decisions?
Treat as **one input** to supervision. Prefer **ranked lists** and **bands** over single binary labels. Compare every release to the **dummy** baseline.

### What should the organization do differently?
Use outputs to **structure questions** in case conference (“what changed in health or school records?”), **not** to replace investigation or legal process.

### What decision does this support?
**Prioritization** of **internal** review time—aligned to a **chosen threshold** from Section 5.

---



## 6. Feature Importance & Operational Insights

The next cell extracts **logistic coefficients** (associational direction) and **Random Forest Gini importances** (split-based contribution). **They can disagree**—use both as **complementary** stories, not as causal effects.

---



In [ ]:
feat_names = log_pipe.named_steps["prep"].get_feature_names_out()
coefs = pd.Series(log_pipe.named_steps["clf"].coef_.ravel(), index=feat_names).sort_values()

plt.figure(figsize=(8, max(5, len(coefs) * 0.18)))
coefs.plot(kind="barh", color=np.where(coefs.values < 0, "#1a9850", "#d73027"))
plt.title("Logistic coefficients (+ → higher log-odds of incident in next 60d)")
plt.tight_layout()
plt.show()

rf = rf_pipe.named_steps["clf"]
imp = pd.Series(rf.feature_importances_, index=feat_names).sort_values(ascending=False)

plt.figure(figsize=(8, max(5, len(imp) * 0.18)))
imp.iloc[::-1].tail(20).plot(kind="barh", color="#4575b4")
plt.title("Random Forest — top 20 feature importances")
plt.tight_layout()
plt.show()

print("Top 10 RF importances:")
print(imp.head(10).round(4).to_string())



### Ranking and plain-English takeaways

Typical patterns (your sample may vary—read **your** charts):
- **Higher prior incident counts** and **severity** usually rank highly: **persistence** of facility-reported events is a strong **statistical** predictor of the **next window**—use for **watchlists**, not for **punitive** labeling.
- **Unresolved incidents** may rank up if **open issues** cluster before new events—interpret as **case-load stress**, not proof that “resolution reduces incidents” without an experiment.
- **Education mean / trend / volatility** can appear if academic stress moves with difficult periods—**confounded** with unmeasured trauma or documentation.
- **Safehouse** one-hot levels capture **site-specific baselines**; discuss with **program leadership** as **operations** insight, not blame.

**What staff can do:** combine **top drivers** with **qualitative** knowledge—e.g., if **low health mean** and **high prior incidents** co-occur, schedule **nursing / psych** touchpoints and **education check-in**; document outcomes for **continuous improvement**.

---



## 7. Causal & Relationship Analysis

### Strongest predictors (associational)
The **prior 90-day incident bundle** (count, severity, unresolved) almost always dominates **marginal** predictive power. **Health and education rolling features** add **incremental** signal in some cohorts—the **forest** can exploit **interactions** (e.g., few prior incidents but **steep negative** education trend).

### Case-management logic (theory)
**Stress accumulation** and **environmental instability** often show up across **multiple record types** simultaneously. A model that **fuses** incidents, wellbeing, and school signals mirrors how **experienced staff** synthesize a case—though staff integrate **unwritten** context the CSVs lack.

### Correlation ≠ causation (explicit)
- **Prior incidents** predict **future incidents** partly because **underlying risk** persists—not because “counting incidents causes” new ones.
- **Risk level** and **case status** are **assigned** labels and **phase indicators**; they **confound** severity, placement decisions, and **who stays Active**.
- **Safehouse** differences **confound** staffing ratios, **cohort mix**, **local stressors**, and **reporting norms**.

### Confounders to name
**Baseline child adversity**, **external family events**, **staffing vacancies**, **under-reporting** vs **over-documentation**, and **time-varying policies** are **not** fully captured. The model learns **patterns in the data as recorded**, not **structural causal effects** of interventions.

### Limits of inference
We **cannot** claim that changing one input **causes** fewer incidents without **randomized** or **quasi-experimental** designs (rare here) or **very strong** assumptions. Coefficients and importances are **evidence of association** in **historical** operations data.

### What is still actionable without causality
- **Watchlist prioritization** under **uncertainty** (decision support).
- **Training** staff on **early multi-domain signals** visible in records.
- **Hypotheses** for **quality improvement** (e.g., timeliness of health follow-up after incidents)—test with **prospective** tracking, not model weights alone.

---



## Key Findings

- **Grouped CV** and **held-out residents** are mandatory sanity checks; compare **always** to **dummy**.
- **Preprocessing in-pipeline** avoids **imputation leakage** from the test split.
- **Threshold** must match **capacity** and **safety philosophy**—report **multiple** cutoffs, not only 0.5.
- **Logistic** for **story**; **RF** for **ranking**; **neither** replaces **mandatory reporting** or **clinical judgment**.

---



## Recommended Actions

- **Legal / ethics** review of internal risk scores and **fairness** across sites and subgroups when sample size allows.
- **Train** staff on **false positives** (dignity-preserving language) and **false negatives** (no complacency).
- **Monitor** **positive rate**, **AUC**, and **calibration** quarterly as new incidents accrue.

---



## 8. Deployment Plan

### What triggers scoring
**Nightly** batch after case notes sync (or weekly if data latency).

### Where results appear
**.NET Case Management → Resident roster** with internal-only `risk_band`; detail view shows **top 3 drivers** in plain language.

### PostgreSQL: `resident_incident_risk`

```sql
CREATE TABLE IF NOT EXISTS resident_incident_risk (
  id BIGSERIAL PRIMARY KEY,
  resident_id INTEGER NOT NULL REFERENCES residents(resident_id),
  as_of_date DATE NOT NULL,
  horizon_days INTEGER NOT NULL,
  incident_risk_probability DOUBLE PRECISION NOT NULL,
  risk_band VARCHAR(16) NOT NULL,
  model_version VARCHAR(32) NOT NULL,
  top_drivers JSONB,
  scored_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
  UNIQUE (resident_id, as_of_date, horizon_days, model_version)
);
```

### Artifacts
Persist the fitted **`Pipeline`** with **`joblib`** under **`machine_learning/ml_pipelines/artifacts/`** (or flat `ml_pipelines/artifacts/` per repo layout).

---



In [ ]:
from joblib import dump

final = clone(rf_pipe)
final.fit(X, y)
dump(final, OUTPUT_DIR / "incident_risk_v3.joblib")
print("Saved:", OUTPUT_DIR / "incident_risk_v3.joblib")

